In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

df = pd.read_csv("loan_fraud_input.csv")

# Example: encode categorical variables
df = pd.get_dummies(df, drop_first=True)

X = df.drop("fraud", axis=1)
y = df["fraud"]

# Initialize k-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Scale and store results
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Store results for each fold
svm_scores = []
xgboost_scores = []
svm_reports = []
xgboost_reports = []

for train_index, test_index in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Apply SMOTE to the training data for each fold
    smote = SMOTE(random_state=42)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

    # SVM Model
    param_grid_svm = {
        'C': [1, 10],
        'kernel': ['linear', 'rbf', 'sigmoid'],
        'gamma': ['scale', 'auto']
    }
    svm_model = GridSearchCV(SVC(probability=True), param_grid_svm, cv=3, scoring='roc_auc')
    svm_model.fit(X_train_smote, y_train_smote)
    y_pred_svm = svm_model.predict(X_test)
    y_proba_svm = svm_model.predict_proba(X_test)[:, 1]
    svm_scores.append(roc_auc_score(y_test, y_proba_svm))
    svm_reports.append(classification_report(y_test, y_pred_svm, output_dict=True))

    # XGBoost Model
    param_grid_xgb = {
        'max_depth': [3, 5],
        'learning_rate': [0.01, 0.1],
        'n_estimators': [50, 100]
    }
    xgb_model = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='logloss'), param_grid_xgb, cv=3, scoring='roc_auc')
    xgb_model.fit(X_train_smote, y_train_smote)
    y_pred_xgb = xgb_model.predict(X_test)
    y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
    xgboost_scores.append(roc_auc_score(y_test, y_proba_xgb))
    xgboost_reports.append(classification_report(y_test, y_pred_xgb, output_dict=True))

# Average scores across folds

print("Average SVM ROC-AUC Score:", np.mean(svm_scores))
print("Average XGBoost ROC-AUC Score:", np.mean(xgboost_scores))

# Example output for the last fold (you can aggregate reports if needed)
print("SVM Confusion Matrix (Last Fold):\n", confusion_matrix(y_test, y_pred_svm))
print("SVM Classification Report (Last Fold):\n", classification_report(y_test, y_pred_svm))
print("XGBoost Confusion Matrix (Last Fold):\n", confusion_matrix(y_test, y_pred_xgb))
print("XGBoost Classification Report (Last Fold):\n", classification_report(y_test, y_pred_xgb))